# PgVector To Kayak Exact Search

This notebook shows the direct Postgres storage path:

1. encode text into late-interaction token vectors
2. store those vectors in Postgres with `pgvector`
3. load an exact filtered slice back into Kayak
4. search that slice locally on the Mojo backend

Set `KAYAK_PGVECTOR_DSN` to a Postgres database where `CREATE EXTENSION vector` works before running the notebook.

Install for this notebook with:

```bash
uv add kayak "psycopg[binary]" pgvector numpy
```

or:

```bash
pixi add --pypi kayak "psycopg[binary]" pgvector numpy
```

In [1]:
import hashlib
import json
import os
import time
import uuid

import numpy as np
import psycopg

import kayak

DSN = os.environ.get("KAYAK_PGVECTOR_DSN")
if not DSN:
    raise RuntimeError("Set KAYAK_PGVECTOR_DSN before executing this notebook.")

BACKEND = kayak.MOJO_EXACT_CPU_BACKEND
BACKEND_INFO = kayak.backend_info(BACKEND)
if not BACKEND_INFO.available:
    raise RuntimeError(BACKEND_INFO.availability_reason)

TABLE_NAME = f"kayak_pgvector_demo_{uuid.uuid4().hex[:8]}"

print("available_backends:", kayak.available_backends())
print("backend_info:", BACKEND_INFO)
print("dsn:", DSN)
print("table_name:", TABLE_NAME)

available_backends: ('numpy_reference', 'mojo_exact_cpu')
backend_info: BackendInfo(name='mojo_exact_cpu', available=True, requires_mojo=True, query_layouts=('nested', 'flat_dim128'), index_layouts=('packed', 'hybrid_flat_dim128'), availability_reason='Kayak can invoke Mojo via: /Users/teilomillet/Code/kayak/.pixi/envs/default/bin/mojo')
dsn: postgresql://postgres@127.0.0.1:55432/postgres
table_name: kayak_pgvector_demo_f2f2849d


In [2]:
def token_vectors(text: str) -> np.ndarray:
    rows = []
    for token in text.lower().split():
        vector = np.zeros(128, dtype=np.float32)
        hot_index = int(hashlib.sha256(token.encode("utf-8")).hexdigest(), 16) % 128
        vector[hot_index] = np.float32(1.0)
        rows.append(vector)
    return np.stack(rows)


def timed_avg_ms(fn, repeats: int = 20) -> float:
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    end = time.perf_counter()
    return round(((end - start) / repeats) * 1000.0, 3)


encoder = kayak.open_encoder(
    "callable",
    query_encoder=token_vectors,
    document_encoder=token_vectors,
)

documents = encoder.encode_documents(
    ["doc-a", "doc-b", "doc-c", "doc-d"],
    [
        "pixi installs python mojo and kayak together",
        "pgvector keeps late interaction rows in postgres",
        "kayak loads the exact slice and searches it locally",
        "metadata filters stay in postgres before kayak loads the slice",
    ],
)

metadata_rows = [
    {"topic": "installation", "tenant": "acme"},
    {"topic": "storage", "tenant": "acme"},
    {"topic": "search", "tenant": "acme"},
    {"topic": "filtering", "tenant": "beta"},
]

query = encoder.encode_query("exact late interaction search")
print("document_count:", documents.document_count)
print("vector_dim:", documents.vector_dim)
print("query_vector_count:", query.vector_count)

document_count: 4
vector_dim: 128
query_vector_count: 4


In [3]:
with psycopg.connect(DSN, autocommit=True) as connection:
    with connection.cursor() as cursor:
        cursor.execute("CREATE EXTENSION IF NOT EXISTS vector")
        cursor.execute(f'DROP TABLE IF EXISTS public."{TABLE_NAME}"')

with kayak.open_store("pgvector", dsn=DSN, table_name=TABLE_NAME) as store:
    store.upsert(documents, metadata=metadata_rows)
    stats = store.stats()
    filtered_index = store.load_index(where={"topic": "search", "tenant": "acme"}, include_text=True)
    hits = kayak.search(query, filtered_index, k=1, backend=BACKEND)
    numpy_hits = kayak.search(query, filtered_index, k=1, backend=kayak.NUMPY_REFERENCE_BACKEND)

summary = {
    "stats": {
        "kind": stats.kind,
        "document_count": stats.document_count,
        "total_vector_count": stats.total_vector_count,
        "vector_dim": stats.vector_dim,
        "has_texts": stats.has_texts,
        "has_metadata": stats.has_metadata,
    },
    "filtered_doc_ids": list(filtered_index.doc_ids),
    "filtered_doc_texts": list(filtered_index.doc_texts or ()),
    "mojo_top_hit": {"doc_id": hits[0].doc_id, "score": round(float(hits[0].score), 3)},
    "numpy_top_hit": {"doc_id": numpy_hits[0].doc_id, "score": round(float(numpy_hits[0].score), 3)},
}

summary

{'stats': {'kind': 'pgvector',
  'document_count': 4,
  'total_vector_count': 33,
  'vector_dim': 128,
  'has_texts': True,
  'has_metadata': True},
 'filtered_doc_ids': ['doc-c'],
 'filtered_doc_texts': ['kayak loads the exact slice and searches it locally'],
 'mojo_top_hit': {'doc_id': 'doc-c', 'score': 2.0},
 'numpy_top_hit': {'doc_id': 'doc-c', 'score': 2.0}}

In [4]:
with psycopg.connect(DSN) as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            f"SELECT pg_typeof(token_matrix)::text, COUNT(*) FROM public.\"{TABLE_NAME}\" GROUP BY 1"
        )
        storage_rows = cursor.fetchall()

storage_rows

[('vector[]', 4)]

In [5]:
def load_filtered_index():
    with kayak.open_store("pgvector", dsn=DSN, table_name=TABLE_NAME) as store:
        return store.load_index(where={"tenant": "acme"}, include_text=True)

candidate_index = load_filtered_index()

def search_exact():
    return kayak.search(query, candidate_index, k=2, backend=BACKEND)


timings = {
    "pgvector_load_index_ms": timed_avg_ms(load_filtered_index, repeats=20),
    "kayak_exact_search_ms": timed_avg_ms(search_exact, repeats=100),
    "loaded_doc_ids": list(candidate_index.doc_ids),
}

timings

{'pgvector_load_index_ms': 4.687,
 'kayak_exact_search_ms': 0.025,
 'loaded_doc_ids': ['doc-a', 'doc-b', 'doc-c']}

In [6]:
with psycopg.connect(DSN, autocommit=True) as connection:
    with connection.cursor() as cursor:
        cursor.execute(f'DROP TABLE IF EXISTS public."{TABLE_NAME}"')

print(json.dumps({"dropped_table": TABLE_NAME}, indent=2))

{
  "dropped_table": "kayak_pgvector_demo_f2f2849d"
}
